# 02_bronze_ingestion — Incremental Raw Ingestion via the Blob SDK

- Lists every file landed under `raw/<source>/` across all 9 sources using
  the Python `azure-storage-blob` SDK (plain HTTPS, not Spark/Auto Loader)
- Skips any blob whose name is already recorded in bronze's
  `source_blob_name` column — that's what makes a re-run only pull
  genuinely new files, in place of an Auto Loader checkpoint
- Stores each file's content **as-is** + lineage columns (`source`, `kind`,
  `source_blob_name`, `ingested_at`)
- No parsing into the shared article schema, no cleaning, no dedup: that's
  silver's job

This is a plain batch step, not a stream — there's no backlog to drain
with `availableNow`, just "list what's there, download what's new, append
it once."

In [0]:
%run ./00_config

# 00_config — Shared Config & Helpers

`%run` by every other notebook. Centralizes the storage connection string, 
source list, table/volume names, and small logging/counting helpers.

**This is the connection-string variant of the pipeline.** Bronze reads 
files landed in ADLS using the plain Python `azure-storage-blob` SDK and a 
storage account connection string — not Auto Loader over an `abfss://` 
path, and not a Unity Catalog External Location. That's a deliberate 
choice, not a step backward: Databricks Serverless blocks 
`spark.conf.set("fs.azure.account.key...")` outright (`CONFIG_NOT_AVAILABLE`), 
which is what Auto Loader over `abfss://` would need without an External 
Location, and the External Location route needs grants (`READ FILES`, 
`WRITE FILES`, `CREATE MANAGED STORAGE`) that may not be available to you 
on a shared/class storage account. The Blob SDK never touches Spark's 
filesystem config at all -- it's a plain HTTPS call, same mechanism the 
live-fetch version (`py_work/`) already uses successfully against public 
APIs on serverless -- so it works regardless of either restriction.

**Note on architecture:** Bronze/Silver/Gold are **Unity Catalog managed 
tables** — `fomoless.bronze.articles_raw`, `fomoless.silver.articles`, 
`fomoless.silver.articles_rejected`, `fomoless.gold.articles`. `01_setup` 
creates the schemas (assumes the `fomoless` catalog itself already exists — 
see the note in `01_setup` if it doesn't yet).

**Bronze is a plain batch step, not a stream** — since there's no Auto 
Loader checkpoint tracking "what's already been read", incrementality 
comes from checking each landed file's blob name against what's already in 
`bronze.articles_raw` (`source_blob_name`) before downloading it, so a 
re-run only pulls genuinely new files. **Silver stays incremental** via its 
own Structured Streaming checkpoint, stored in a small Unity Catalog 
**managed Volume** (`VOLUME_PATH`, created by `01_setup`) — Databricks-managed 
storage under the catalog you already own, so it needs no extra grants 
either. That Volume also holds nothing else; dedup-merge audit logs go 
straight to ADLS via the same Blob SDK connection as bronze.

---
**SECURITY FIX (this revision):** a previous version of this notebook had 
a live Azure Storage account key hardcoded as the `connection_string` 
widget's default value. That key has been removed from this file — **if 
you're seeing this note, rotate that key in Azure Portal (Storage Account 
→ Access keys → Rotate) if you haven't already**, since it was exposed in 
a shared notebook. Going forward, set the connection string via the widget 
at run time or the `AZURE_STORAGE_CONNECTION_STRING` environment variable 
— never as a hardcoded default.

**New in this revision:** `CANONICAL_SOURCE_NAMES` (CHK-4 enum gate), 
`SOURCES_WITHOUT_NATIVE_SUMMARY` (CHK-5 conditional-completeness exception 
list), and `CLOCK_SKEW_LOG_PREFIX` (CHK-7 audit trail) — all consumed by 
`03_silver_transform`.

---
**Gold/Mongo revision:** added `ensure_cdf_enabled` (moved here from 
`03_silver_transform` since `04_gold_aggregation` now needs the identical 
logic — shared, not duplicated), `TREND_DECAY_HALF_LIFE_DAYS` and 
`MOMENTUM_MULTIPLIERS` (TR-11/TR-13), new `gold`/`mongo_sink` checkpoint 
paths, and `get_mongo_uri()` (mirrors `get_widget_or_env` — never 
hardcoded).

**⚠️ FLAGGED GAP — TR-13's momentum multiplier table is incomplete.** The 
source Transformation Rules sheet is cut off after the `BBC`/`The Guardian` 
→ `3.0` branch; the multiplier values for dev.to, Medium, Towards Data 
Science, Hacker News, Stack Overflow Blog, Stack Exchange, and KDnuggets 
were never visible in the source image. `MOMENTUM_MULTIPLIERS` below only 
has the one confirmed entry — every other source falls back to 
`DEFAULT_MOMENTUM_MULTIPLIER = 1.0` as an explicit placeholder. **This dict 
needs real values from the business before `04_gold_aggregation`'s trend 
scores are correct**, not just structurally complete.

## 1. Serverless "spark not initialized" workaround

On some Databricks Serverless sessions, the notebook-provided `spark` 
global isn't ready the first time a cell references it, raising 
`SystemError: Internal error: spark should be initialized with the first 
notebook command.` Explicitly building a session via `databricks.connect` 
fixes it and is a no-op anywhere `spark` is already fine.

## 2. Storage connection string

From Azure Portal → your storage account → **Access keys** → Connection 
string (key 1 or key 2, either works). Settable via a Databricks widget 
(`connection_string`) or the `AZURE_STORAGE_CONNECTION_STRING` environment 
variable -- never hardcode it directly in a cell.

## 3. Sources & schema

## 4. Catalog, schemas, tables & volume

## 5. Gold trend-scoring constants (TR-11, TR-13)

## 6. MongoDB connection (for `06_load_mongodb`)

Same pattern as the storage connection string above: widget first, then 
environment variable, never a hardcoded default. `MONGO_URI` should 
already contain the target database name (e.g. 
`.../fomoless?retryWrites=true&w=majority`); `MONGO_COLLECTION_NAME` names 
the collection within it.

## 7. Helpers

[2026-09-25 20:01:45 UTC] Config loaded — container: fomoless, catalog: fomoless


In [0]:
from azure.storage.blob import BlobServiceClient
from pyspark.sql import functions as F

## 1. Find out what's already in bronze

In [0]:
if spark.catalog.tableExists(TABLES["bronze"]):
    already_ingested = {
        row.source_blob_name
        for row in spark.table(TABLES["bronze"]).select("source_blob_name").distinct().collect()
    }
else:
    already_ingested = set()

log(f"{len(already_ingested)} blob(s) already in bronze — those will be skipped")

[2026-09-25 20:01:50 UTC] 0 blob(s) already in bronze — those will be skipped


## 2. Download every new file under `raw/<source>/`

In [0]:
container_client = BlobServiceClient.from_connection_string(CONNECTION_STRING).get_container_client(
    CONTAINER_NAME
)

rows = []
fetch_summary = []
for src in SOURCES:
    folder, kind = src["folder"], src["kind"]
    prefix = f"{RAW_PREFIX}/{folder}/"
    blobs = list(container_client.list_blobs(name_starts_with=prefix))
    # skip zero-byte "directory placeholder" blobs some tools create, and
    # anything already ingested in a previous run
    new_blobs = [b for b in blobs if b.size > 0 and b.name not in already_ingested]

    fetched_this_source = 0
    for blob in new_blobs:
        try:
            blob_client = container_client.get_blob_client(blob.name)
            content = blob_client.download_blob().readall().decode("utf-8")
            ingested_at = blob.last_modified  # tz-aware UTC datetime from the SDK
            rows.append({
                "source": folder, "kind": kind, "raw_content": content,
                "source_blob_name": blob.name, "ingested_at": ingested_at,
            })
            fetched_this_source += 1
        except Exception as exc:
            log(f"⚠️  {folder}: failed to download '{blob.name}': {exc}")

    fetch_summary.append((folder, len(blobs), len(new_blobs), fetched_this_source))
    log(f"{folder}: {len(blobs)} file(s) in ADLS, {len(new_blobs)} new, {fetched_this_source} downloaded")

display(
    spark.createDataFrame(
        fetch_summary, "source STRING, total_in_adls INT, new_since_last_run INT, downloaded INT"
    )
)

[2026-09-25 20:01:52 UTC] devto: 15 file(s) in ADLS, 15 new, 15 downloaded
[2026-09-25 20:01:52 UTC] medium: 15 file(s) in ADLS, 15 new, 15 downloaded
[2026-09-25 20:01:53 UTC] towardsdatascience: 15 file(s) in ADLS, 15 new, 15 downloaded
[2026-09-25 20:01:53 UTC] bbc: 15 file(s) in ADLS, 15 new, 15 downloaded
[2026-09-25 20:01:55 UTC] hackernews: 26 file(s) in ADLS, 26 new, 26 downloaded
[2026-09-25 20:01:55 UTC] stackoverflow_blog: 15 file(s) in ADLS, 15 new, 15 downloaded
[2026-09-25 20:01:56 UTC] stackexchange: 15 file(s) in ADLS, 15 new, 15 downloaded
[2026-09-25 20:01:56 UTC] kdnuggets: 0 file(s) in ADLS, 0 new, 0 downloaded
[2026-09-25 20:01:56 UTC] guardian: 0 file(s) in ADLS, 0 new, 0 downloaded


source,total_in_adls,new_since_last_run,downloaded
devto,15,15,15
medium,15,15,15
towardsdatascience,15,15,15
bbc,15,15,15
hackernews,26,26,26
stackoverflow_blog,15,15,15
stackexchange,15,15,15
kdnuggets,0,0,0
guardian,0,0,0


## 3. Append every new row to the bronze Delta table

In [0]:
if rows:
    spark_new = spark.createDataFrame(pd.DataFrame(rows))
    spark_new.write.format("delta").mode("append").option("mergeSchema", "true").saveAsTable(TABLES["bronze"])
    log(f"Appended {len(rows)} new row(s) to {TABLES['bronze']}")
else:
    log("No new files to ingest this run.")

[2026-09-25 20:02:12 UTC] Appended 116 new row(s) to fomoless.bronze.articles_raw


In [0]:
log(f"{TABLES['bronze']} total rows: {row_count(TABLES['bronze'])}")

if spark.catalog.tableExists(TABLES["bronze"]):
    display(
        spark.table(TABLES["bronze"])
        .groupBy("source")
        .agg(F.count("*").alias("records"), F.max("ingested_at").alias("last_ingested_at"))
        .orderBy("source")
    )

[2026-09-25 20:02:13 UTC] fomoless.bronze.articles_raw total rows: 116


source,records,last_ingested_at
bbc,15,2026-09-20T03:06:44.000Z
devto,15,2026-09-20T03:05:40.000Z
hackernews,26,2026-09-20T03:07:07.000Z
medium,15,2026-09-20T03:06:02.000Z
stackexchange,15,2026-09-20T03:07:52.000Z
stackoverflow_blog,15,2026-09-20T03:07:29.000Z
towardsdatascience,15,2026-09-20T03:06:24.000Z
